# Finetuning

In [1]:
!pip install groq

Prompting

In [2]:
import os
from groq import Groq

In [3]:
groq_key = os.environ.get("GROQ_API_KEY")
client = None
if not groq_key:
    print("GROQ_API_KEY not set; skipping GROQ model calls. Set GROQ_API_KEY to enable them.")
else:
    client = Groq(api_key=groq_key)
    def prompt_based_query(user_query):
      response = client.chat.completions.create(
          model="llama-3.3-70b-versatile", 
          messages=[
              {"role":"system","content":"You are an advocate."},
              {"role":"user","content":f'answer the query {user_query}'}
          ],
          temperature=0.2
      )
      return response.choices[0].message.content

    print(prompt_based_query("what is the law against spammers?"))

ValueError: Set GROQ_API_KEY in your environment before using GROQ.

## RAG

Load Document

In [ ]:
!pip install langchain faiss-cpu groq langchain_community tiktoken

   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/552.2 kB ? eta -:--:--
   ------------------ --------------------- 262.1/552.2 kB ? eta -:--:--
   ------------------ --------------------- 262.1/552.2 kB ? eta -:--:--
  

In [ ]:
#step 1 load document
try:
    import pandas as pd
except Exception as e:
    raise ImportError('pandas is required. Install with `pip install pandas`') from e
csv_path = r'D:\vs code programs\AIML\AGENTIC_AI\prompting-rag-finetuning-KanwalKishoreSece\archive\legal_text_classification.csv'
df = pd.read_csv(csv_path)
# If the CSV has a 'text' column use it, otherwise join all columns per row
if 'text' in df.columns:
    texts = df['text'].astype(str).tolist()
else:
    texts = df.astype(str).apply(lambda r: ' '.join(r.values), axis=1).tolist()
class SimpleDoc:
    def __init__(self, text, idx=None):
        self.page_content = text
        self.metadata = {}
        self.id = str(idx) if idx is not None else None

# create documents with ids
documents = [SimpleDoc(t, idx=i) for i, t in enumerate(texts)]

<>:3: SyntaxWarning: invalid escape sequence '\A'
<>:3: SyntaxWarning: invalid escape sequence '\A'
C:\Users\vpsel\AppData\Local\Temp\ipykernel_22756\3425905755.py:3: SyntaxWarning: invalid escape sequence '\A'
  loader=TextLoader('D:\vs code programs\AIML\AGENTIC_AI\prompting-rag-finetuning-KanwalKishoreSece\archive\legal_text_classification.csv')


In [ ]:
#step 2 create the embedding + vector DB
import os
from langchain_community.vectorstores import FAISS

# Try OpenAI embeddings first if OPENAI_API_KEY is set
openai_api_key = os.environ.get("OPENAI_API_KEY")
embeddings = None
if openai_api_key:
    try:
        from langchain_community.embeddings import OpenAIEmbeddings
        embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
        print('Using OpenAIEmbeddings')
    except Exception as e:
        print('OpenAIEmbeddings not available:', e)
# Fallback to sentence-transformers if OpenAI embeddings are not available
if embeddings is None:
    try:
        from sentence_transformers import SentenceTransformer
        class SBEmbeddings:
            def __init__(self, model_name='all-MiniLM-L6-v2'):
                self.model = SentenceTransformer(model_name)
            def embed_documents(self, texts):
                embs = self.model.encode(texts, show_progress_bar=False)
                return [list(map(float, e)) for e in embs]
        embeddings = SBEmbeddings()
        print('Using sentence-transformers embeddings')
    except Exception as e:
        raise ValueError('No embedding provider available. Set OPENAI_API_KEY or install sentence-transformers.')

vector_db = FAISS.from_documents(documents, embeddings)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
#step 3 Retrieval + Generation using GROQ
def groq_query(groq_query_string):
    if client is None:
        raise ValueError("GROQ client is not initialized. Set GROQ_API_KEY to run GROQ queries.")
    response = client.groq.query(
        model="groq-1.0",
        query=groq_query_string
    )
    return response

if client is not None:
    print(groq_query('*[_type == "company"]{name, description, _id}'))
else:
    print("Skipping GROQ query because GROQ_API_KEY is not set.")

NameError: name 'client' is not defined

In [ ]:
if client is not None:
    print(groq_query("what is our company refund policy"))
else:
    print("Skipping GROQ query because GROQ_API_KEY is not set.")

ABC Retail's refund policy allows customers to request a refund within 7 days of delivery. Once the refund is approved, it is processed within 5-7 business days.


# Fine Tuning(PEFT +LoRA)

In [5]:
!pip install transformers datasets peft accelerate bitsandbytes

#step 1 : Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import bitsandbytes as bnb
import os
model_name = os.environ.get('HF_MODEL', 'facebook/opt-1.3b')
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map='auto',
    )
    print('Loaded model', model_name)
except Exception as e:
    print('Failed to load', model_name, ':', e)
    # fallback to a smaller Hugging Face model
    fallback = 'gpt2'
    tokenizer = AutoTokenizer.from_pretrained(fallback)
    model = AutoModelForCausalLM.from_pretrained(fallback)
    print('Fell back to', fallback)

In [6]:
#step 2 : Apply LoRA(PEFT)
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
#preapre the model for k bit trining
model = prepare_model_for_kbit_training(model)
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

d:\vs code programs\AIML\AGENTIC_AI\prompting-rag-finetuning-KanwalKishoreSece\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'model' is not defined

In [7]:
#step 3: dataset preperation
from datasets import Dataset
data=[
    {"text":"Q: what is ROI?\nA: Return on Investment is a probability metric"},
    {"text":"Q: Define Churn?\nA: Percentage of customer leaving the services"}
]
dataset=Dataset.from_list(data)

In [8]:
#step 4: tokenization
def tokenize_function(example):
  return tokenizer(example["text"],truncation=True,padding="max_length")
tokenized_dataset=dataset.map(tokenize_function)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]


NameError: name 'tokenizer' is not defined

In [9]:
#step 5: Training
from transformers import Trainer , TrainingArguments
#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)
training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

NameError: name 'tokenized_dataset' is not defined

In [ ]:
#step 6: inferences
def generate_response(prompt):
  inputs=tokenizer(prompt,return_tensors="pt").to("cuda")
  outputs=model.generate(**inputs,max_new_tokens=100)
  return tokenizer.decode(outputs[0])

In [ ]:
print(generate_response("what is ROI"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype

<s> what is ROI and the same time, and the same time to the same time to the same time to the world.















































































